## Scraping Ukrant headlines

In [28]:
# Import libraries...

from selenium import webdriver # for web scraping, as the page is dynamically loaded with JavaScript
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup # for parsing the HTML
import json
import time

In [29]:
# Set up Selenium WebDriver and navigate to the page

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.get("https://ukrant.nl/categorie/nieuws-en/?lang=en")
time.sleep(3)

In [30]:
# Click "Load More" until all articles are loaded
while True:
    try:
        load_more = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.CLASS_NAME, "td_ajax_load_more")) # find the "Load More" button by its class name
        )
        driver.execute_script("arguments[0].scrollIntoView();", load_more) # scroll to the button to ensure it's in view
        load_more.click() # click the button
        print("Clicked Load More...")
        time.sleep(2)
    except:
        print("No more pages.")
        break # If the button is not found or not clickable, we assume we've loaded all articles

Clicked Load More...
Clicked Load More...
Clicked Load More...
Clicked Load More...
Clicked Load More...
Clicked Load More...
Clicked Load More...
Clicked Load More...
Clicked Load More...
Clicked Load More...
Clicked Load More...
Clicked Load More...
Clicked Load More...
Clicked Load More...
No more pages.


In [31]:
# Scrape all loaded headlines
soup = BeautifulSoup(driver.page_source, "html.parser")
driver.quit()

In [32]:
print(soup.prettify())

<html class="td-md-is-os-x td-md-is-chrome" lang="en-US">
 <!--<![endif]-->
 <head>
  <title>
   News Archives - UKrant.nl
  </title>
  <link data-rocket-prefetch="" href="https://fonts.googleapis.com" rel="dns-prefetch"/>
  <link data-rocket-prefetch="" href="https://www.googletagmanager.com" rel="dns-prefetch"/>
  <link data-rocket-prefetch="" href="https://c0.wp.com" rel="dns-prefetch"/>
  <link data-rocket-prefetch="" href="https://i0.wp.com" rel="dns-prefetch"/>
  <link data-rocket-prefetch="" href="https://stats.wp.com" rel="dns-prefetch"/>
  <link as="style" data-rocket-preload="" href="https://fonts.googleapis.com/css?family=Open%20Sans%3A400%2C600%2C700%7CRoboto%3A400%2C600%2C700&amp;display=swap" rel="preload"/>
  <link as="font" crossorigin="" data-rocket-preload="" href="https://ukrant.nl/wp-content/themes/Newspaper-child/fonts/BerninaSans-Web-NarrowRegular.woff" rel="preload"/>
  <link as="font" crossorigin="" data-rocket-preload="" href="https://ukrant.nl/wp-content/theme

In [33]:
all_headlines = []
for h3 in soup.find_all("h3", class_="entry-title"): # find all h3 elements with class "entry-title"
    a = h3.find("a") # find the <a> tag inside the h3, which contains the headline text and URL
    if a:
        all_headlines.append({
            "title": a.get_text(strip=True), # get the headline text
            "url": a.get("href", "") # get the URL from the href attribute, default to empty string if not found
        })

In [34]:
with open("ukrant_headlines.json", "w", encoding="utf-8") as f:
    json.dump(all_headlines, f, ensure_ascii=False, indent=2) # save the headlines to a JSON file with pretty formatting

print(f"Saved {len(all_headlines)} headlines to ukrant_headlines.json")

Saved 171 headlines to ukrant_headlines.json
